## 04 - Test the /predict endpoint

Load a real experiment from the training CSV, reshape it into the API payload
(timestamps + Z/W/X values dict), POST it to the running FastAPI server, and
compare the predicted titer to the recorded final titer.

Start the server (from the project root):

```bash
uv run -m uvicorn src.api:app --reload
```

In [10]:
import pandas as pd
import requests

In [ ]:
API_URL = "http://localhost:8000"
TRAIN_DATA_PATH = "../data/datahow_interview_train_data.csv"
TRAIN_TARGETS_PATH = "../data/datahow_interview_train_targets.csv"
EXP = "Exp 1"

In [12]:
requests.get(f"{API_URL}/health").json()

{'status': 'ok'}

### Load the data and pick one experiment

In the raw CSV, `Z:` setpoints are only filled on day 0 and `NaN` everywhere
else; `W:` and `X:` columns are populated at every timestep.

In [13]:
train_df = pd.read_csv(TRAIN_DATA_PATH).drop(columns="RowID")
targets_df = pd.read_csv(TRAIN_TARGETS_PATH).drop(columns="RowID")

exp_df = train_df.query("Exp == @EXP").sort_values("Time[day]").reset_index(drop=True)
exp_df.head()

,Exp,Time[day],Z:FeedStart,Z:FeedEnd,Z:FeedRateGlc,Z:FeedRateGln,Z:phStart,Z:phEnd,Z:phShift,Z:tempStart,...,W:temp,W:pH,W:FeedGlc,W:FeedGln,X:VCD,X:Glc,X:Gln,X:Amm,X:Lac,X:Lysed
0,Exp 2,0,1.0,12.0,5.69697,6.979798,6.606061,6.207071,8.0,37.121212,...,37.121212,6.606061,0.00000,0.000000,1.714640,4.514751,4.870449,0.100000,0.100000,0.0
1,Exp 2,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,37.121212,6.606061,5.69697,6.979798,3.001574,3.783190,3.096066,0.163976,1.612478,0.0
2,Exp 2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,37.121212,6.606061,5.69697,6.979798,4.726490,8.649033,7.994568,0.412594,2.484435,0.0
3,Exp 2,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,37.121212,6.606061,5.69697,6.979798,6.752777,12.360235,11.593952,1.341968,4.319551,0.0
4,Exp 2,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,37.121212,6.606061,5.69697,6.979798,9.071930,17.601629,15.572984,1.772807,5.940848,0.0


### Build the request payload

Per `inference_server_spec.yml`:
- `Z:` keys must be single-element arrays (we take the day-0 value)
- `W:` and `X:` keys must be arrays whose length matches `timestamps`

In [14]:
z_cols = [c for c in exp_df.columns if c.startswith("Z:")]
w_cols = [c for c in exp_df.columns if c.startswith("W:")]
x_cols = [c for c in exp_df.columns if c.startswith("X:")]

day0 = exp_df.loc[exp_df["Time[day]"] == 0].iloc[0]

values = {col: [float(day0[col])] for col in z_cols}
for col in w_cols + x_cols:
    values[col] = exp_df[col].astype(float).tolist()

payload = {
    "timestamps": exp_df["Time[day]"].astype(float).tolist(),
    "values": values,
}

print(f"timestamps: {payload['timestamps']}")
print(f"# Z keys: {len(z_cols)}, # W keys: {len(w_cols)}, # X keys: {len(x_cols)}")

timestamps: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
# Z keys: 13, # W keys: 4, # X keys: 6


### Call the endpoint and compare to the recorded titer

In [15]:
response = requests.post(f"{API_URL}/predict", json=payload)
response.raise_for_status()
result = response.json()
result

{'titer': 1298.0020843510417}

In [16]:
actual = float(targets_df.loc[targets_df["Exp"] == EXP, "Y:Titer"].iloc[0])
predicted = float(result["titer"])
print(f"Experiment: {EXP}")
print(f"  predicted titer: {predicted:.2f}")
print(f"  actual titer:    {actual:.2f}")
print(f"  abs error:       {abs(predicted - actual):.2f}")
print(f"  rel error:       {abs(predicted - actual) / actual:.2%}")

Experiment: Exp 2
  predicted titer: 1298.00
  actual titer:    1278.06
  abs error:       19.94
  rel error:       1.56%
